# Silver `avaliacoes.csv` / `clickstream.csv`

## 1. Importações e Configurações de Parâmetros

In [0]:
import re
import time
from datetime import datetime
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalogo", "workspace")
catalogo = dbutils.widgets.get("catalogo")

spark.sql(f"USE CATALOG {catalogo}")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

tabelas_escopo = ["tb_clickstream", "tb_avaliacoes"]

print(f"Catálogo em uso: {catalogo}")
print(f"Tabelas no escopo deste notebook: {tabelas_escopo}")

## 2. Tabela de auditoria `silver.dq_log`
Histórico persistente das execuções do pipeline na camada Silver.
Segue a mesma estrutura da Bronze para permitir rastreio fim a fim.

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS silver.dq_log (
        execucao_id      STRING,
        camada           STRING,
        tabela           STRING,
        etapa            STRING,
        registros        BIGINT,
        duracao_segundos DOUBLE,
        origem           STRING,
        timestamp_log    TIMESTAMP
    ) USING DELTA
""")

print("Tabela silver.dq_log pronta")

## 3. Transformação — `tb_clickstream`

Regras aplicadas nesta função:

1. **Deduplicação:** Remove eventos duplicados por `id_evento`, mantendo o registro com `timestamp_ingestion` mais recente.
2. **Tipagem:** `data_evento` → timestamp, `tempo_pagina_seg` → inteiro. Usa `try_cast` para converter falhas em nulo sem quebrar o pipeline.
3. **Saneamento de tempo:** Valores zero ou negativos em `tempo_pagina_seg` são anulados — tempo não mensurável não deve entrar nas médias da Gold.
4. **Canal:** Canonizado para `Web`, `App` ou `Mobile`. Sem padronização, a Gold criaria grupos duplicados ao calcular `canal_preferido` no `gold_cliente_360`.
5. **Dispositivo:** Canonizado para `celular`, `computador` ou `tablet`.
6. **Tipo de evento:** 40+ variações brutas normalizadas para 8 valores canônicos alinhados com o contrato da Gold: `visualizacao_pagina`, `busca`, `visualizacao_produto`, `adicao_carrinho`, `login`, `pagamento`, `compra`, `abandono_carrinho`.
7. **Origem da sessão:** Canonizado para português explícito. O valor `email`, ausente no código anterior, foi adicionado.

In [0]:
def transformar_clickstream(df: DataFrame) -> DataFrame:

    # 1. Deduplicação pelo id_evento mais recente
    window_click = Window.partitionBy("id_evento").orderBy(F.col("timestamp_ingestion").desc())

    return (
        df.withColumn("row_num", F.row_number().over(window_click))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
        .replace("null", None)

        # 2. Tipagem
        .withColumn("data_evento",       F.expr("try_cast(data_evento as timestamp)"))
        .withColumn("tempo_pagina_seg",  F.expr("try_cast(tempo_pagina_seg as int)"))

        # 3. Sanea tempo: zero ou negativo vira nulo
        .withColumn("tempo_pagina_seg",
            F.when(F.col("tempo_pagina_seg") <= 0, None)
             .otherwise(F.col("tempo_pagina_seg"))
        )

        # 4. Canonização de canal → Web / Mobile / App
        .withColumn("canal",
            F.when(F.lower(F.col("canal")).isin("web", "w"), "Web")
             .when(F.lower(F.col("canal")).isin("app", "aplicativo"), "App")
             .when(F.lower(F.col("canal")).isin("mobile_web", "mobile web"), "Mobile")
             .otherwise("desconhecido")
        )

        # 5. Canonização de dispositivo → celular / computador / tablet
        .withColumn("dispositivo",
            F.when(F.lower(F.col("dispositivo")).isin("mob", "mobile", "celular"), "celular")
             .when(F.lower(F.col("dispositivo")).isin("desktop", "computador"),"computador")
             .when(F.lower(F.col("dispositivo")).isin("tablet", "tab"),"tablet")
             .otherwise("desconhecido")
        )

        # 6. Canonização de tipo_evento — alinhado com o contrato da Gold
        .withColumn("tipo_evento",
            F.when(F.lower(F.col("tipo_evento")).isin(
                "page_view", "pageview", "page-view", "pv", "visualizacao_pagina"
            ), "visualizacao_pagina")

             .when(F.lower(F.col("tipo_evento")).isin(
                "search", "busca", "srch"
             ), "busca")

             .when(F.lower(F.col("tipo_evento")).isin(
                "product_view", "product-view", "productview", "prod_view", "visualizacao_produto"
             ), "visualizacao_produto")

             .when(F.lower(F.col("tipo_evento")).isin(
                "add_to_cart", "addtocart", "add-to-cart", "adicionar", "adicao_carrinho", "adicionar_carrinho"
             ), "adicao_carrinho")

             .when(F.lower(F.col("tipo_evento")).isin(
                "login", "log_in", "signin"
             ), "login")

             .when(F.lower(F.col("tipo_evento")).isin(
                "checkout", "check_out", "pagamento"
             ), "pagamento")

             .when(F.lower(F.col("tipo_evento")).isin(
                "purchase", "buy", "compra"
             ), "compra")

             .when(F.lower(F.col("tipo_evento")).isin(
                "abandon_cart", "abandon-cart", "abandono", "abandoncart"
             ), "abandono_carrinho")

             .otherwise("desconhecido")
        )

        # 7. Canonização de origem_sessao → português explícito
        .withColumn("origem_sessao",
            F.when(F.col("origem_sessao") == "organic", "organico")
             .when(F.col("origem_sessao") == "paid_search", "busca_paga")
             .when(F.col("origem_sessao") == "direct", "direto")
             .when(F.col("origem_sessao") == "social", "social")
             .when(F.col("origem_sessao") == "email", "email")
             .otherwise("desconhecido")
        )
    )

print("transformar_clickstream definida")

## 4. Transformação — `tb_avaliacoes`

Regras aplicadas nesta função:

1. **Deduplicação:** Remove avaliações duplicadas por `id_avaliacao`, mantendo o registro mais recente.
2. **Tipagem:** `data_avaliacao` → timestamp, `nota_nps` → inteiro via `try_cast`.
3. **`nota_produto`:** O campo chega como texto qualitativo ("ótimo", "ruim") **ou** como número — ambos os casos são tratados. Após a conversão, valores fora de `[1, 5]` são anulados. Sem essa validação, valores como `0` e `-1` contaminariam as médias em `gold_produto_performance` e `gold_cliente_360`.
4. **`nota_nps`:** Validada para o range `[0, 10]`. Valores fora do intervalo são anulados para não distorcer o `nps_medio_avaliacoes_cliente` do `gold_cliente_360`.
5. **`recomenda`:** Múltiplas variações textuais ("sim", "s", "yes", "y", "1") convertidas para booleano.
6. **`perfil_nps`:** Derivado da `nota_nps` já validada — Promotor (≥9), Neutro (7–8), Detrator (0–6).
7. **`sentimento`:** Campo exigido pelo contrato `gold_avaliacoes`. Derivado de `nota_produto` validada: nota 4–5 → positivo, nota 3 → neutro, nota 1–2 → negativo. Se `nota_produto` for nulo, `sentimento` também fica nulo.

In [0]:
def transformar_avaliacoes(df: DataFrame) -> DataFrame:

    # 1. Deduplicação pelo id_avaliacao mais recente
    window_aval = Window.partitionBy("id_avaliacao").orderBy(F.col("timestamp_ingestion").desc())

    return (
        df.withColumn("row_num", F.row_number().over(window_aval))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
        .replace("null", None)

        # 2. Tipagem
        .withColumn("data_avaliacao", F.expr("try_cast(data_avaliacao as timestamp)"))
        .withColumn("nota_nps",       F.expr("try_cast(nota_nps as int)"))

        # 3. nota_produto: texto qualitativo → inteiro, depois valida range [1,5]
        .withColumn("nota_produto",
            F.when(F.lower(F.col("nota_produto")).isin("péssimo", "pessimo"), 1)
             .when(F.lower(F.col("nota_produto")) == "ruim",                  2)
             .when(F.lower(F.col("nota_produto")) == "regular",               3)
             .when(F.lower(F.col("nota_produto")) == "bom",                   4)
             .when(F.lower(F.col("nota_produto")).isin("ótimo", "otimo"),     5)
             .otherwise(F.expr("try_cast(nota_produto as int)"))
        )
        # Anula qualquer valor fora do range válido [1, 5]
        .withColumn("nota_produto",
            F.when(F.col("nota_produto").between(1, 5), F.col("nota_produto"))
             .otherwise(None)
        )

        # 4. nota_nps: valida range [0, 10] — anula fora do range
        .withColumn("nota_nps",
            F.when(F.col("nota_nps").between(0, 10), F.col("nota_nps"))
             .otherwise(None)
        )

        # 5. recomenda → booleano
        .withColumn("recomenda",
            F.when(F.lower(F.col("recomenda")).isin("s", "sim", "yes", "y", "1"), True)
             .when(F.lower(F.col("recomenda")).isin("n", "nao", "não", "no", "0"), False)
             .otherwise(None)
        )

        # 6. perfil_nps derivado da nota_nps já validada
        .withColumn("perfil_nps",
            F.when(F.col("nota_nps") >= 9,                                     "Promotor")
             .when(F.col("nota_nps").between(7, 8),                            "Neutro")
             .when(F.col("nota_nps").between(0, 6),                            "Detrator")
             .otherwise(None)
        )

        # 7. sentimento derivado de nota_produto — exigido pelo contrato gold_avaliacoes
        .withColumn("sentimento",
            F.when(F.col("nota_produto") >= 4, "positivo")
             .when(F.col("nota_produto") == 3, "neutro")
             .when(F.col("nota_produto") <= 2, "negativo")
             .otherwise(None)
        )
    )

print("transformar_avaliacoes definida")

## 5. Função Orquestradora da Silver

Padrão único aplicado às duas tabelas:

- **Leitura da Bronze:** Sempre a partir da tabela Delta materializada, nunca direto do CSV.
- **Roteamento:** Cada tabela tem sua própria função de transformação. Qualquer tabela fora do escopo lança exceção explícita.
- **`timestamp_silver`:** Injetado após a transformação, registra quando o dado foi promovido à Silver.
- **Escrita com `overwriteSchema=true`:** Permite evolução de schema sem quebrar o pipeline.
- **Contagem pós-escrita:** Feita via `spark.table` para aproveitar as estatísticas do transaction log Delta, sem re-scan do dado.
- **Auditoria:** Cada execução é registrada em `silver.dq_log` com tabela, volumetria, duração e origem.

In [0]:
def processar_silver(tabela: str, execucao_id: str) -> tuple[int, float]:
    inicio = time.time()

    df_bronze = spark.table(f"bronze.{tabela}")

    if tabela == "tb_clickstream":
        df_silver = transformar_clickstream(df_bronze)
    elif tabela == "tb_avaliacoes":
        df_silver = transformar_avaliacoes(df_bronze)
    else:
        raise ValueError(f"Nenhuma regra de negócio definida na Silver para: {tabela}")

    df_silver = df_silver.withColumn("timestamp_silver", F.current_timestamp())

    (
        df_silver.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"silver.{tabela}")
    )

    registros = spark.table(f"silver.{tabela}").count()
    duracao   = time.time() - inicio
    origem    = f"bronze.{tabela}"

    log_entry = spark.createDataFrame(
        [(execucao_id, "silver", tabela, "limpeza_enriquecimento", registros, duracao, origem, datetime.now())],
        ["execucao_id", "camada", "tabela", "etapa", "registros", "duracao_segundos", "origem", "timestamp_log"]
    )
    log_entry.write.format("delta").mode("append").saveAsTable("silver.dq_log")

    return registros, duracao

print("Função orquestradora da Silver definida")

## 6. Execução

In [0]:
execucao_id = datetime.now().strftime("%Y%m%d_%H%M%S")
resultados  = {}

for tabela in tabelas_escopo:
    qtd, duracao = processar_silver(tabela, execucao_id)
    resultados[tabela] = qtd
    print(f"silver.{tabela:20s}  {qtd:>9,} registros processados em {duracao:>6.2f}s")

print(f"\nTotal salvo na Silver: {sum(resultados.values()):,} registros em {len(resultados)} tabelas")
print(f"Execução registrada em silver.dq_log com id: {execucao_id}")

## 7. Quality Gate da Camada Silver

Validação executada ao final do notebook. Lança exceção em caso de falha para interromper o Workflow antes que a Gold consuma dados inconsistentes.

As regras são aplicadas por tabela:

1. **Existência:** A tabela deve estar presente no catálogo — cobre escritas silenciosamente abortadas por erro de permissão ou rede.
2. **Volumetria:** Nenhuma tabela pode estar vazia — captura o caso em que os filtros de limpeza descartaram 100% dos registros.
3. **Colunas obrigatórias:** Cada tabela tem seu próprio conjunto de colunas exigidas, alinhado com o que a Gold vai consumir.
4. **Range de `nota_produto` (avaliações):** Confirma que nenhum valor fora de `[1, 5]` escapou da transformação.
5. **Range de `nota_nps` (avaliações):** Confirma que nenhum valor fora de `[0, 10]` escapou da transformação.

In [0]:
print("\nIniciando Quality Gate da camada Silver...")

falhas = []

colunas_obrigatorias_por_tabela = {
    "tb_clickstream": {"timestamp_silver", "tipo_evento", "canal", "dispositivo", "origem_sessao"},
    "tb_avaliacoes":  {"timestamp_silver", "nota_produto", "nota_nps", "sentimento", "perfil_nps", "recomenda"},
}

for tabela in tabelas_escopo:
    nome_completo = f"silver.{tabela}"

    # Regra 1: existência
    if not spark.catalog.tableExists(nome_completo):
        print(f"[ERRO] {nome_completo} não encontrada no catálogo")
        falhas.append(f"{nome_completo} ausente")
        continue

    df    = spark.table(nome_completo)
    total = df.count()

    # Regra 2: volumetria
    if total == 0:
        print(f"[ERRO] {nome_completo} está vazia após processamento")
        falhas.append(f"{nome_completo} vazia")
        continue

    # Regra 3: colunas obrigatórias
    ausentes = colunas_obrigatorias_por_tabela[tabela] - set(df.columns)
    if ausentes:
        print(f"[ERRO] {nome_completo} sem colunas obrigatórias: {ausentes}")
        falhas.append(f"{nome_completo} sem {ausentes}")
        continue

    # Regra 4: nota_produto fora de range [1,5] não pode existir
    if tabela == "tb_avaliacoes":
        fora_range_produto = df.filter(
            F.col("nota_produto").isNotNull() & ~F.col("nota_produto").between(1, 5)
        ).count()
        if fora_range_produto > 0:
            print(f"[ERRO] {nome_completo} contém {fora_range_produto} registros com nota_produto fora de [1,5]")
            falhas.append(f"{nome_completo} nota_produto com valores inválidos")

        # Regra 5: nota_nps fora de range [0,10] não pode existir
        fora_range_nps = df.filter(
            F.col("nota_nps").isNotNull() & ~F.col("nota_nps").between(0, 10)
        ).count()
        if fora_range_nps > 0:
            print(f"[ERRO] {nome_completo} contém {fora_range_nps} registros com nota_nps fora de [0,10]")
            falhas.append(f"{nome_completo} nota_nps com valores inválidos")

    print(f"[OK]   {nome_completo}: {total:,} registros validados")

if falhas:
    raise Exception("Quality Gate da Silver falhou:\n- " + "\n- ".join(falhas))

print("\n[SUCESSO] Camada Silver validada. Dados prontos para a Gold!")